# Setup and Configure NemoClaw on the Brev instance

This notebook is meant to run **on the Brev instance itself**. It shows how to set up or reuse a **NemoClaw** sandbox, configure the **OpenShell** provider for NVIDIA Endpoints, copy **VSS skills** into the OpenClaw workspace, and update OpenClaw UI **`allowedOrigins`**.

**Default launchable:**  
[VSS + NemoClaw launchable](https://brev.nvidia.com/launchable/deploy/now?launchableID=env-3BgcwbtTMrB4IXdnaeDwaq5ULti)

If you are using that launchable, the VM is expected to already have the main prerequisites in place. If not, use the preflight cells below to confirm what is missing.

**What the script does now**

1. If the sandbox **already exists**, it **skips** NemoClaw onboard/install.
2. Otherwise it installs NemoClaw or runs NemoClaw onboard non-interactively.
3. Configures the OpenShell provider for NVIDIA Endpoints.
4. Applies the VSS sandbox policy.
5. Uploads repo `skills/` into the sandbox workspace backing store.
6. Updates OpenClaw UI `allowedOrigins`.

**Security:** prefer `NVIDIA_API_KEY` from environment or Brev secrets. Do **not** commit notebook outputs that contain credentials.


## 1. Settings

<span style="color:red"><strong>Important:</strong> set <code>NVIDIA_API_KEY</code> here or via the environment. <code>NVIDIA_API_KEY</code> is required to run the steps below.</span>

In [ ]:
import os
from pathlib import Path

# ============= Set your NVIDIA API Key here (leave blank to use the existing environment variable)==============
NVIDIA_API_KEY = ""
# ===============================================================================================================


# No need to change these unless you want to customize the script.
NVIDIA_API_KEY = os.environ.get("NVIDIA_API_KEY") or NVIDIA_API_KEY
VSS_REPO_DIR = Path(os.environ.get("VSS_REPO_DIR", "/home/ubuntu/video-search-and-summarization")).resolve()
NEMOCLAW_INIT_SCRIPT_PATH = VSS_REPO_DIR / "deployments" / "scripts" / "nemoclaw" / "init_nemoclaw.sh"
NEMOCLAW_INSTALL = Path("/home/ubuntu/NemoClaw/install.sh")
POLICY_PATH = VSS_REPO_DIR / "assets" / "vss_nemoclaw_policy.yaml"
OPENCLAW_CONFIG_UPDATE_SCRIPT = VSS_REPO_DIR / "deployments" / "scripts" / "nemoclaw" / "update_openclaw_config.py"
SKILLS_DIR = VSS_REPO_DIR / "skills"
NEMOCLAW_SANDBOX_NAME = os.environ.get("NEMOCLAW_SANDBOX_NAME", "demo")
LAUNCHABLE_URL = (
    "https://brev.nvidia.com/launchable/deploy/now"
    "?launchableID=env-3BgcwbtTMrB4IXdnaeDwaq5ULti"
)

print("Sandbox:", NEMOCLAW_SANDBOX_NAME)
print("Launchable:", LAUNCHABLE_URL)
print("\nVSS_REPO_DIR:", VSS_REPO_DIR)
print("NEMOCLAW_INIT_SCRIPT_PATH:", NEMOCLAW_INIT_SCRIPT_PATH)
print("NEMOCLAW_INSTALL:", NEMOCLAW_INSTALL)
print("POLICY_PATH:", POLICY_PATH)
print("OPENCLAW_CONFIG_UPDATE_SCRIPT:", OPENCLAW_CONFIG_UPDATE_SCRIPT)
print("SKILLS_DIR:", SKILLS_DIR)

## 2. Preflight

Run the next cell to confirm the expected keys, files, commands are present on the instance.


In [ ]:
import os
import shutil
from pathlib import Path

RED = "\033[31m"
RESET = "\033[0m"
GREEN = "\033[32m"

checks = {
    "init_nemoclaw.sh": NEMOCLAW_INIT_SCRIPT_PATH.is_file(),
    "NemoClaw install.sh": NEMOCLAW_INSTALL.is_file() and os.access(NEMOCLAW_INSTALL, os.X_OK),
    "vss_nemoclaw_policy.yaml": POLICY_PATH.is_file(),
    "update_openclaw_config.py": OPENCLAW_CONFIG_UPDATE_SCRIPT.is_file(),
    "skills/": SKILLS_DIR.is_dir(),
    "docker": shutil.which("docker") is not None,
    "openshell": shutil.which("openshell") is not None,
    "python3": shutil.which("python3") is not None,
    "NVIDIA_API_KEY set": bool(NVIDIA_API_KEY.strip()),
}

for label, ok in checks.items():
    status = "OK " if ok else "NO "
    color = GREEN if ok else RED
    print(f"{color}{status}{RESET} {label}")

## 3. Install and Configure NemoClaw (OpenClaw) for VSS skills

The next cell runs the script in the foreground so you can watch progress.

The script will:

- create or update the NemoClaw sandbox,
- configure the OpenShell provider,
- apply the VSS NemoClaw policy,
- install VSS skills,
- update the OpenClaw UI `allowedOrigins` setting.

In [ ]:
import os
import subprocess
import time
from collections import deque
from pathlib import Path

init_path = Path(NEMOCLAW_INIT_SCRIPT_PATH)
if not init_path.is_file():
    raise FileNotFoundError(f"Missing init script: {init_path}")

if not NVIDIA_API_KEY.strip():
    raise RuntimeError(
        "NVIDIA_API_KEY is required. Set it in the settings cell or export it before running this cell."
    )

if not POLICY_PATH.is_file():
    raise FileNotFoundError(f"Missing policy file: {POLICY_PATH}")

if not OPENCLAW_CONFIG_UPDATE_SCRIPT.is_file():
    raise FileNotFoundError(f"Missing OpenClaw config update script: {OPENCLAW_CONFIG_UPDATE_SCRIPT}")

env = os.environ.copy()
env["VSS_REPO_DIR"] = str(VSS_REPO_DIR)
env["NEMOCLAW_SANDBOX_NAME"] = NEMOCLAW_SANDBOX_NAME
env["NVIDIA_API_KEY"] = NVIDIA_API_KEY.strip()
env["NEMOCLAW_POLICY_FILE"] = str(POLICY_PATH)
env["OPENCLAW_CONFIG_UPDATE_SCRIPT"] = str(OPENCLAW_CONFIG_UPDATE_SCRIPT)

timeout_sec = 3600
last_output_lines = deque(maxlen=40)
start = time.monotonic()

print("Running:", init_path, flush=True)
print("Applying policy file:", POLICY_PATH, flush=True)
print("Using OpenClaw config update script:", OPENCLAW_CONFIG_UPDATE_SCRIPT, flush=True)
process = subprocess.Popen(
    ["bash", str(init_path)],
    env=env,
    cwd=str(init_path.parent),
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)

try:
    if process.stdout is None:
        raise RuntimeError("Failed to capture init_nemoclaw.sh output")

    for line in process.stdout:
        print(line, end="", flush=True)
        last_output_lines.append(line.rstrip("\n"))
        if time.monotonic() - start > timeout_sec:
            process.kill()
            process.wait()
            raise TimeoutError(f"init_nemoclaw.sh timed out after {timeout_sec} seconds")
finally:
    if process.stdout is not None:
        process.stdout.close()

returncode = process.wait()
if returncode != 0:
    tail = "\n".join(last_output_lines)
    raise RuntimeError(
        f"init_nemoclaw.sh exited with {returncode}\n"
        f"Last output:\n{tail}"
    )
print("Done.", flush=True)


## 4. Verify sandbox, policy, and skills

Rerun the next cell after the install cell. It performs a live verification pass and prints a fresh UTC timestamp so you can tell the output is current.

The next cell checks:

- whether the sandbox exists,
- the current active sandbox policy metadata,
- the expected local policy path and version,
- whether VSS skills landed in `/sandbox/.openclaw-data/workspace/skills`.


In [ ]:
from datetime import datetime, timezone
import json
import re
import shlex
import socket
import subprocess
from pathlib import Path


def run(cmd, check=False, echo=True):
    print("$", shlex.join(cmd))
    r = subprocess.run(cmd, capture_output=True, text=True, check=check)
    if echo:
        if r.stdout:
            print(r.stdout)
        if r.stderr:
            print(r.stderr)
    return r


print("Verification time (UTC):", datetime.now(timezone.utc).strftime("%Y-%m-%d %H:%M:%S %Z"))
print("Sandbox:", NEMOCLAW_SANDBOX_NAME)
print("Expected policy file:", POLICY_PATH)

policy_text = Path(POLICY_PATH).read_text()
policy_version_match = re.search(r"^version:\s*(\d+)", policy_text, re.MULTILINE)
print("Expected local policy version:", policy_version_match.group(1) if policy_version_match else "unknown")

sandbox_result = run(["openshell", "sandbox", "get", NEMOCLAW_SANDBOX_NAME], echo=False)
sandbox_summary = "\n".join(part for part in (sandbox_result.stdout, sandbox_result.stderr) if part)
sandbox_summary = re.sub(r"\x1b\[[0-9;]*m", "", sandbox_summary)
phase_match = re.search(r"Phase:\s+(.+)", sandbox_summary)
namespace_match = re.search(r"Namespace:\s+(.+)", sandbox_summary)
sandbox_id_match = re.search(r"Id:\s+(.+)", sandbox_summary)
print("Sandbox namespace:", namespace_match.group(1).strip() if namespace_match else "unknown")
print("Sandbox phase:", phase_match.group(1).strip() if phase_match else "unknown")
print("Sandbox id:", sandbox_id_match.group(1).strip() if sandbox_id_match else "unknown")

policy_result = run(["openshell", "policy", "get", NEMOCLAW_SANDBOX_NAME])

policy_summary = "\n".join(part for part in (policy_result.stdout, policy_result.stderr) if part)
status_match = re.search(r"Status:\s+(.+)", policy_summary)
active_match = re.search(r"Active:\s+(.+)", policy_summary)
hash_match = re.search(r"Hash:\s+([0-9a-f]+)", policy_summary)
print("Active policy status:", status_match.group(1).strip() if status_match else "unknown")
print("Active policy version:", active_match.group(1).strip() if active_match else "unknown")
print("Active policy hash:", hash_match.group(1) if hash_match else "unknown")

gateway_container = subprocess.run(
    ["docker", "ps", "--format", "{{.Names}}"], capture_output=True, text=True, check=True
).stdout.splitlines()
gateway_container = next((name for name in gateway_container if name.startswith("openshell-cluster-")), None)
print("Gateway container:", gateway_container)

if gateway_container:
    print("VSS workspace skills:")
    skills_cmd = [
        "sudo",
        "docker",
        "exec",
        gateway_container,
        "kubectl",
        "exec",
        "-n",
        "openshell",
        NEMOCLAW_SANDBOX_NAME,
        "-c",
        "agent",
        "--",
        "sh",
        "-lc",
        'su - sandbox -c "openclaw skills list --json"',
    ]
    print("$", shlex.join(skills_cmd))
    skills_result = subprocess.run(skills_cmd, capture_output=True, text=True, check=True)
    skills_text = skills_result.stdout.strip() or skills_result.stderr.strip()
    skills_payload = json.loads(skills_text)
    skills = skills_payload["skills"] if isinstance(skills_payload, dict) else skills_payload
    workspace_skills = [skill for skill in skills if skill.get("source") == "openclaw-workspace"]
    if workspace_skills:
        for skill in workspace_skills:
            print(f"- {skill['name']}")
    else:
        print("No openclaw-workspace skills found.")
else:
    print("Could not determine the OpenShell gateway container; workspace skills were not checked.")

## 5. Verify host reachability from inside the sandbox

<span style="color:red"><strong>Important:</strong> make sure VSS is already deployed on the host before running this step.</span>

Run the next cell after section 4 if you want to confirm the sandbox can reach host services through `host.openshell.internal`.

This checks the policy-allowed host ports and helps distinguish between:

- policy or DNS issues, and
- a host service that simply is not listening on the probed port.


In [ ]:
import shlex
import subprocess

HOST_PORTS = (8000, 9902, 30888, 5601, 9200, 8081)


def run(cmd, check=False):
    print("$", shlex.join(cmd))
    r = subprocess.run(cmd, capture_output=True, text=True, check=check)
    if r.stdout:
        print(r.stdout)
    if r.stderr:
        print(r.stderr)
    return r


gateway_container = subprocess.run(
    ["docker", "ps", "--format", "{{.Names}}"], capture_output=True, text=True, check=True
).stdout.splitlines()
gateway_container = next((name for name in gateway_container if name.startswith("openshell-cluster-")), None)
print("Gateway container:", gateway_container)

if not gateway_container:
    raise RuntimeError("Could not determine the OpenShell gateway container for host reachability checks.")

host_probe_script = f"""
if command -v python3 >/dev/null 2>&1; then
  python3 - <<'PY'
import socket
ports = {list(HOST_PORTS)}
for port in ports:
    s = socket.socket()
    s.settimeout(3)
    try:
        s.connect((\"host.openshell.internal\", port))
    except Exception as exc:
        print(f\"{{port}}: FAIL {{exc}}\")
    else:
        print(f\"{{port}}: OK\")
    finally:
        s.close()
PY
else
  echo \"python3 not available in sandbox for host probe\"
  exit 1
fi
""".strip()

print("Host service reachability from inside the sandbox:")
run(
    [
        "sudo",
        "docker",
        "exec",
        gateway_container,
        "kubectl",
        "exec",
        "-n",
        "openshell",
        NEMOCLAW_SANDBOX_NAME,
        "--",
        "sh",
        "-lc",
        host_probe_script,
    ]
)
print(
    "Note: 'Connection refused' means the sandbox reached host.openshell.internal, "
    "but nothing is listening on that host/port combination."
)


## 6. Open the UI and smoke test it

Run the next cell to print a fresh **OpenClaw UI** link for the current Brev environment.

### Test 1. Open the OpenClaw UI

Use the **OpenClaw UI** link printed by the next cell.

Open that link in your browser.

In [ ]:
import os
import re
import subprocess


def read_etc_environment():
    env = {}
    try:
        with open("/etc/environment", encoding="utf-8") as fp:
            for raw in fp:
                line = raw.strip()
                if not line or line.startswith("#") or "=" not in line:
                    continue
                key, value = line.split("=", 1)
                env[key.strip()] = value.strip().strip('"').strip("'")
    except OSError:
        return env
    return env


gateway_container = subprocess.run(
    ["docker", "ps", "--format", "{{.Names}}"], capture_output=True, text=True, check=True
).stdout.splitlines()
gateway_container = next((name for name in gateway_container if name.startswith("openshell-cluster-")), None)
print("Gateway container:", gateway_container)

if not gateway_container:
    raise RuntimeError("Could not determine the OpenShell gateway container; no UI link generated.")

openclaw_ui_url = None
env_id = os.environ.get("BREV_ENV_ID", "").strip() or read_etc_environment().get("BREV_ENV_ID", "").strip()
if env_id:
    origin = f"https://openclaw0-{env_id}.brevlab.com"
    token_result = subprocess.run(
        [
            "sudo",
            "docker",
            "exec",
            gateway_container,
            "kubectl",
            "exec",
            "-n",
            "openshell",
            NEMOCLAW_SANDBOX_NAME,
            "--",
            "sh",
            "-lc",
            'su - sandbox -c "openclaw dashboard"',
        ],
        capture_output=True,
        text=True,
    )
    token_output = "\n".join(part for part in (token_result.stdout, token_result.stderr) if part)
    match = re.search(r"/#token=([0-9a-fA-F]+)", token_output)
    openclaw_ui_url = f"{origin}/#token={match.group(1)}" if match else origin
    print("OpenClaw UI:", openclaw_ui_url)
else:
    print(
        "Could not resolve Brev environment ID. Set BREV_ENV_ID in the environment "
        "or add it to /etc/environment."
    )

### Test 2. Verify the LLM works in chat

Start a chat with the agent and send a simple prompt such as:

- `hello`
- `what model are you using?`
- `list your available skills`

![OpenClaw UI link screenshot](./images/OpenClawUIChat.png)

You should get a normal model response back. If the UI opens but chat fails, re-check provider setup and the active policy.

### Test 3. Verify the VSS skills are imported

Open the **Skills** tab in the UI and confirm the **VSS skills** are present.

![OpenClaw UI link screenshot](./images/OpenClawUISkills.png)

If the UI opens and chat works but the Skills tab is missing VSS skills, re-check `/sandbox/.openclaw-data/workspace/skills` in the verify step above.

## 7. Uninstall NemoClaw

In [ ]:
import subprocess

cmd = "curl -fsSL https://raw.githubusercontent.com/NVIDIA/NemoClaw/refs/heads/main/uninstall.sh | bash -s -- --yes"
print(f"$ {cmd}")
subprocess.run(["bash", "-lc", cmd], check=True)